<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Chapter3_Joins_and_Join_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3 — Joins & Join Algorithms
## การเชื่อมข้อมูลจากหลายตารางเข้าด้วยกัน

**หัวข้อ**
1. หลักการของ JOIN — เงื่อนไข ON, Alias, Cross Join ที่ควรระวัง
2. INNER JOIN & JOIN หลายตาราง — เครื่องมือหลักในการสร้าง feature table
3. OUTER JOIN: LEFT / RIGHT / FULL — เครื่องมือหลักของการทำ EDA เพื่อหา 'ข้อมูลที่ขาดหาย'
4. Join Algorithms — Nested loop, Hash, Sort-merge
5. เทคนิค JOIN ขั้นสูง — Self-join, Non-equi join, Multi-condition


In [ ]:
# ── ติดตั้งและเตรียม DuckDB ──
!pip install duckdb --quiet


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')   # in-memory เหมาะกับการฝึก

def run(sql):
    """รัน SQL แล้วคืนผลเป็น DataFrame"""
    try:
        return con.execute(sql).df()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

print("พร้อมใช้งาน DuckDB", duckdb.__version__)


พร้อมใช้งาน DuckDB 1.3.2


---
## เตรียมข้อมูลตัวอย่าง

ใช้ฐานข้อมูลห้องสมุด 3 ตาราง: `member` (สมาชิก), `book` (หนังสือ), `loan` (การยืม) ตลอดทั้ง notebook
ซึ่งจะมีข้อมูล **สมาชิกที่ไม่เคยยืมหนังสือ** และ **หนังสือที่ไม่เคยถูกยืม** เพื่อฝึกเรื่อง OUTER JOIN และ NULL


In [ ]:
# ตาราง member — มี 3 คนที่ไม่เคยยืมหนังสือเลย (ปิยะ, กมล, สุดา)
con.execute("""
CREATE TABLE member (member_id INTEGER PRIMARY KEY, name VARCHAR, city VARCHAR)
""")
con.execute("""INSERT INTO member VALUES
    (1, 'สมชาย', 'กรุงเทพฯ'), (2, 'สมหญิง', 'กรุงเทพฯ'), (3, 'สมศักดิ์', 'กรุงเทพฯ'),
    (4, 'สมศรี', 'เชียงใหม่'), (5, 'สมพร', 'เชียงใหม่'),
    (6, 'วิชัย', 'ชลบุรี'), (7, 'อรุณ', 'ชลบุรี'),
    (8, 'พรทิพย์', 'ขอนแก่น'), (9, 'ธนกร', 'ขอนแก่น'),
    (10, 'ปิยะ', 'สงขลา'), (11, 'กมล', 'ภูเก็ต'), (12, 'สุดา', 'อุบลราชธานี')
""")

# ตาราง book — มี 3 เล่มที่ไม่เคยถูกยืมเลย
con.execute("""
CREATE TABLE book (book_id INTEGER PRIMARY KEY, title VARCHAR)
""")
con.execute("""INSERT INTO book VALUES
    (101, 'คิดแบบผู้ชนะ'), (102, 'โลกของโซฟี'), (103, 'เศรษฐศาสตร์แบบเข้าใจง่าย'),
    (104, 'นิยายรักในสวนดอกไม้'), (105, 'คู่มือ Data Science'), (106, 'ประวัติศาสตร์ไทย'),
    (107, 'จิตวิทยาการทำงาน'), (108, 'สอน Python เบื้องต้น'), (109, 'ปรัชญาเบื้องต้น'),
    (110, 'บันทึกนักเดินทาง'), (111, 'อาหารไทยโบราณ'), (112, 'ดาราศาสตร์เบื้องต้น')
""")

# ตาราง loan — ทุกแถวมี member_id และ book_id ที่ถูกต้องเสมอ
con.execute("""
CREATE TABLE loan (loan_id INTEGER PRIMARY KEY, member_id INTEGER, book_id INTEGER, loan_date DATE, status VARCHAR)
""")
con.execute("""INSERT INTO loan VALUES
    (1,1,101,'2024-01-05','returned'), (2,1,102,'2024-02-10','returned'),
    (3,2,101,'2024-01-20','returned'), (4,2,103,'2024-03-01','borrowed'),
    (5,3,101,'2024-04-15','returned'), (6,3,104,'2024-04-20','borrowed'),
    (7,4,105,'2024-01-10','returned'), (8,4,101,'2024-05-01','returned'),
    (9,5,106,'2024-02-05','returned'),
    (10,6,107,'2024-03-10','borrowed'), (11,6,108,'2024-03-15','returned'),
    (12,7,109,'2024-01-25','returned'), (13,7,101,'2024-06-01','borrowed'),
    (14,8,102,'2024-02-20','returned'), (15,8,105,'2024-05-10','borrowed'),
    (16,9,103,'2024-04-05','returned'), (17,9,106,'2024-06-15','returned'),
    (18,1,109,'2024-07-01','borrowed'), (19,2,108,'2024-07-05','returned'),
    (20,5,101,'2024-07-10','returned')
""")

print("สร้างตารางสำเร็จ")
print("member มี", con.execute("SELECT count(*) FROM member").fetchone()[0], "แถว")
print("book มี", con.execute("SELECT count(*) FROM book").fetchone()[0], "แถว")
print("loan มี", con.execute("SELECT count(*) FROM loan").fetchone()[0], "แถว")
run("SELECT * FROM member LIMIT 5")


สร้างตารางสำเร็จ
member มี 12 แถว
book มี 12 แถว
loan มี 20 แถว


,member_id,name,city
0,1,สมชาย,กรุงเทพฯ
1,2,สมหญิง,กรุงเทพฯ
2,3,สมศักดิ์,กรุงเทพฯ
3,4,สมศรี,เชียงใหม่
4,5,สมพร,เชียงใหม่


---
# 1. หลักการของ JOIN

JOIN คือการนำตารางมาเชื่อมกัน โดยจับคู่แถวที่มีค่าตรงกันในคอลัมน์ที่กำหนด (เงื่อนไข `ON`)
ส่วนใหญ่เป็นการจับคู่ **Foreign key** ของตารางหนึ่ง กับ **Primary key** ของอีกตาราง


### 1.1 โครงสร้าง JOIN พื้นฐาน · Alias

In [ ]:
# จับคู่แถวใน loan กับ member ที่ member_id ตรงกัน เมื่อ l และ m คือ alias
run("""
SELECT l.loan_id, m.name, l.book_id
FROM loan l
JOIN member m
  ON l.member_id = m.member_id
ORDER BY l.loan_id
""")

,loan_id,name,book_id
0,1,สมชาย,101
1,2,สมชาย,102
2,3,สมหญิง,101
3,4,สมหญิง,103
4,5,สมศักดิ์,101
5,6,สมศักดิ์,104
6,7,สมศรี,105
7,8,สมศรี,101
8,9,สมพร,106
9,10,วิชัย,107


### 1.2 ลืมเงื่อนไข ON → Cross Join
ถ้าลืมเงื่อนไข `ON` (หรือใช้ comma แทน) จะได้ **ผลคูณไขว้ (cross join)** — ทุกแถวของตารางหนึ่งจับคู่กับทุกแถวของอีกตาราง


In [ ]:
# มี ON — ได้ผลลัพธ์เท่ากับจำนวน loan จริง (20 แถว)
print("มี ON:", len(run("SELECT m.name, l.loan_id FROM member m JOIN loan l ON m.member_id = l.member_id")), "แถว")

# ลืม ON — ได้ผลคูณไขว้ 12 (member) x 20 (loan) = 240 แถว!
print("ลืม ON:", len(run("SELECT m.name, l.loan_id FROM member m, loan l")), "แถว (ผิด! คูณไขว้ทุกคู่)")


มี ON: 20 แถว
ลืม ON: 240 แถว (ผิด! คูณไขว้ทุกคู่)


### แบบฝึกหัด 1
1. เขียน query เชื่อม `loan` กับ `book` เพื่อแสดงชื่อหนังสือที่ถูกยืมแต่ละรายการ (แสดง loan_id, title, status) พร้อมตั้ง alias ให้ทั้งสองตาราง
2. ลองเขียน query เชื่อม `book` กับ `member` โดย**ไม่ใส่เงื่อนไข ON** แล้วนับว่าได้กี่แถว (ควรได้เท่ากับ 12 × 12)
3. แก้ query ข้อ 2 ให้ถูกต้องด้วยการเชื่อมผ่าน `loan` แทน (book JOIN loan ON book_id, loan JOIN member ON member_id)


In [ ]:
# เขียนโค้ดข้อ 1 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1

,loan_id,title,status
0,1,คิดแบบผู้ชนะ,returned
1,2,โลกของโซฟี,returned
2,3,คิดแบบผู้ชนะ,returned
3,4,เศรษฐศาสตร์แบบเข้าใจง่าย,borrowed
4,5,คิดแบบผู้ชนะ,returned
5,6,นิยายรักในสวนดอกไม้,borrowed
6,7,คู่มือ Data Science,returned
7,8,คิดแบบผู้ชนะ,returned
8,9,ประวัติศาสตร์ไทย,returned
9,10,จิตวิทยาการทำงาน,borrowed


In [ ]:
# เขียนโค้ดข้อ 2 ตรงนี้
print(len(run("...............")), "แถว")

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2

144 แถว


In [ ]:
# เขียนโค้ดข้อ 3 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3

,title,name
0,คิดแบบผู้ชนะ,สมชาย
1,โลกของโซฟี,สมชาย
2,คิดแบบผู้ชนะ,สมหญิง
3,เศรษฐศาสตร์แบบเข้าใจง่าย,สมหญิง
4,คิดแบบผู้ชนะ,สมศักดิ์
5,นิยายรักในสวนดอกไม้,สมศักดิ์
6,คู่มือ Data Science,สมศรี
7,คิดแบบผู้ชนะ,สมศรี
8,ประวัติศาสตร์ไทย,สมพร
9,จิตวิทยาการทำงาน,วิชัย


---
# 2. INNER JOIN & JOIN หลายตาราง

INNER JOIN คืนเฉพาะแถวที่จับคู่ได้ทั้งสองตาราง — รูปแบบที่พบบ่อยที่สุด
เป็นเครื่องมือหลักในการสร้าง **feature table** สำหรับงาน Machine Learning


### 2.1 JOIN สามตาราง

In [ ]:
# ใครยืมหนังสือเรื่องอะไร — เชื่อม loan + member + book พร้อมกัน
run("""
SELECT m.name, b.title, l.status
FROM loan l
JOIN member m ON l.member_id = m.member_id
JOIN book b ON l.book_id = b.book_id
ORDER BY l.loan_id
""")


,name,title,status
0,สมชาย,คิดแบบผู้ชนะ,returned
1,สมชาย,โลกของโซฟี,returned
2,สมหญิง,คิดแบบผู้ชนะ,returned
3,สมหญิง,เศรษฐศาสตร์แบบเข้าใจง่าย,borrowed
4,สมศักดิ์,คิดแบบผู้ชนะ,returned
5,สมศักดิ์,นิยายรักในสวนดอกไม้,borrowed
6,สมศรี,คู่มือ Data Science,returned
7,สมศรี,คิดแบบผู้ชนะ,returned
8,สมพร,ประวัติศาสตร์ไทย,returned
9,วิชัย,จิตวิทยาการทำงาน,borrowed


### 2.2 JOIN + WHERE กรองผลเพิ่มเติม

In [ ]:
# หารายการที่ยังไม่คืน (status = 'borrowed') พร้อมชื่อสมาชิกและหนังสือ
run("""
SELECT m.name, b.title, l.status
FROM loan l
JOIN member m ON l.member_id = m.member_id
JOIN book b ON l.book_id = b.book_id
WHERE l.status = 'borrowed'
""")


,name,title,status
0,อรุณ,คิดแบบผู้ชนะ,borrowed
1,สมหญิง,เศรษฐศาสตร์แบบเข้าใจง่าย,borrowed
2,สมศักดิ์,นิยายรักในสวนดอกไม้,borrowed
3,พรทิพย์,คู่มือ Data Science,borrowed
4,วิชัย,จิตวิทยาการทำงาน,borrowed
5,สมชาย,ปรัชญาเบื้องต้น,borrowed


### แบบฝึกหัด 2
1. JOIN สามตาราง แสดงเฉพาะรายการยืมของสมาชิกที่อยู่เมือง `'เชียงใหม่'`
2. นับจำนวนแถวทั้งหมดที่ INNER JOIN ของ `loan JOIN member` คืนมา แล้วอธิบายว่าทำไมถึงได้ตัวเลขนี้
3. หารายการที่ `status = 'returned'` พร้อมชื่อสมาชิกและชื่อหนังสือ เรียงตาม loan_date จากเก่าไปใหม่


In [ ]:
# เขียนโค้ดข้อ 1 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1

,name,city,title,status
0,สมพร,เชียงใหม่,คิดแบบผู้ชนะ,returned
1,สมศรี,เชียงใหม่,คู่มือ Data Science,returned
2,สมพร,เชียงใหม่,ประวัติศาสตร์ไทย,returned
3,สมศรี,เชียงใหม่,คิดแบบผู้ชนะ,returned


In [ ]:
# เขียนโค้ดข้อ 2 ตรงนี้
print(con.execute("............................").fetchone()[0])

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2

20


In [ ]:
# เขียนโค้ดข้อ 3 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3

,name,title,loan_date,status
0,สมชาย,คิดแบบผู้ชนะ,2024-01-05,returned
1,สมศรี,คู่มือ Data Science,2024-01-10,returned
2,สมหญิง,คิดแบบผู้ชนะ,2024-01-20,returned
3,อรุณ,ปรัชญาเบื้องต้น,2024-01-25,returned
4,สมพร,ประวัติศาสตร์ไทย,2024-02-05,returned
5,สมชาย,โลกของโซฟี,2024-02-10,returned
6,พรทิพย์,โลกของโซฟี,2024-02-20,returned
7,วิชัย,สอน Python เบื้องต้น,2024-03-15,returned
8,ธนกร,เศรษฐศาสตร์แบบเข้าใจง่าย,2024-04-05,returned
9,สมศักดิ์,คิดแบบผู้ชนะ,2024-04-15,returned


---
# 3. OUTER JOIN: LEFT / RIGHT / FULL

เมื่อต้องการเก็บแถวที่จับคู่ไม่ได้ไว้ด้วย ใช้ **OUTER JOIN** — แถวที่ไม่มีคู่จะได้ค่า **NULL** ในฝั่งอีกตาราง
เป็นเครื่องมือหลักของการทำ **EDA** เพื่อหา "ข้อมูลที่ขาดหาย"


### 3.1 LEFT JOIN — เก็บทุกแถวจากตารางซ้าย

In [ ]:
# สมาชิกที่ไม่เคยยืมหนังสือ (LEFT JOIN + IS NULL)
run("""
SELECT l.loan_id, m.name
FROM member m
LEFT JOIN loan l ON m.member_id = l.member_id
WHERE l.loan_id IS NULL
""")


,loan_id,name
0,<NA>,กมล
1,<NA>,ปิยะ
2,<NA>,สุดา


### 3.2 RIGHT JOIN — เก็บทุกแถวจากตารางขวา

In [ ]:
# หนังสือที่ไม่เคยถูกยืม — ใช้ RIGHT JOIN (loan อยู่ซ้าย, book อยู่ขวา)
run("""
SELECT l.loan_id, b.book_id, b.title
FROM loan l
RIGHT JOIN book b ON l.book_id = b.book_id
WHERE l.loan_id IS NULL
""")


,loan_id,book_id,title
0,<NA>,110,บันทึกนักเดินทาง
1,<NA>,112,ดาราศาสตร์เบื้องต้น
2,<NA>,111,อาหารไทยโบราณ


### 3.3 FULL JOIN — เก็บทุกแถวจากทั้งสองฝั่ง

In [ ]:
# FULL JOIN ระหว่าง member กับ loan
run("""
SELECT *
FROM member m
FULL JOIN loan l ON m.member_id = l.member_id
ORDER BY m.member_id
""")


,member_id,name,city,loan_id,member_id_1,book_id,loan_date,status
0,1,สมชาย,กรุงเทพฯ,1,1,101,2024-01-05,returned
1,1,สมชาย,กรุงเทพฯ,2,1,102,2024-02-10,returned
2,1,สมชาย,กรุงเทพฯ,18,1,109,2024-07-01,borrowed
3,2,สมหญิง,กรุงเทพฯ,3,2,101,2024-01-20,returned
4,2,สมหญิง,กรุงเทพฯ,4,2,103,2024-03-01,borrowed
5,2,สมหญิง,กรุงเทพฯ,19,2,108,2024-07-05,returned
6,3,สมศักดิ์,กรุงเทพฯ,5,3,101,2024-04-15,returned
7,3,สมศักดิ์,กรุงเทพฯ,6,3,104,2024-04-20,borrowed
8,4,สมศรี,เชียงใหม่,7,4,105,2024-01-10,returned
9,4,สมศรี,เชียงใหม่,8,4,101,2024-05-01,returned


### แบบฝึกหัด 3
1. หาสมาชิกที่ไม่เคยยืมหนังสือ ที่อยู่เมือง `'ภูเก็ต'` หรือ `'สงขลา'`
2. ใช้ `LEFT JOIN` เพื่อหาหนังสือที่ไม่เคยถูกยืม
3. เทียบจำนวนแถวจาก `INNER JOIN` กับ `LEFT JOIN` ของ `member` กับ `loan` แล้วอธิบายว่าทำไมตัวเลขถึงต่างกัน


In [ ]:
# เขียนโค้ดข้อ 1 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1

,name,city
0,กมล,ภูเก็ต
1,ปิยะ,สงขลา


In [ ]:
# เขียนโค้ดข้อ 2 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2

,book_id,title
0,110,บันทึกนักเดินทาง
1,112,ดาราศาสตร์เบื้องต้น
2,111,อาหารไทยโบราณ


In [ ]:
# เขียนโค้ดข้อ 3 ตรงนี้
inner_n = con.execute("...........................").fetchone()[0]
left_n  = con.execute("...........................").fetchone()[0]
print("INNER JOIN:", inner_n, " | LEFT JOIN:", left_n)

In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3

INNER JOIN: 20  | LEFT JOIN: 23


---
# 4. Join Algorithms

เบื้องหลัง ฐานข้อมูลใช้วิธีใดวิธีหนึ่งใน 3 แบบเพื่อจับคู่แถว:

| Algorithm | หลักการ | เหมาะกับ |
|---|---|---|
| **Nested Loop** | ไล่ทีละแถวของตารางหนึ่ง เทียบกับทุกแถวของอีกตาราง | ตารางเล็ก หรือมี index |
| **Hash Join** | สร้าง hash table จากตารางเล็กก่อน แล้วจับคู่ตารางใหญ่ผ่าน hash | ข้อมูลไม่เรียงลำดับ |
| **Sort-Merge** | เรียงลำดับทั้งสองตารางตามคีย์ก่อน แล้วไล่จับคู่พร้อมกัน | ข้อมูลถูกเรียงอยู่แล้ว |

ไม่ต้องจำรายละเอียด แต่ควรรู้ว่า **ฐานข้อมูลเลือก algorithm ให้เองอัตโนมัติ** ตามขนาดข้อมูลและ index ที่มี — ดูได้ผ่านคำสั่ง `EXPLAIN`


### 4.1 ดู algorithm ด้วย EXPLAIN

In [ ]:
# ดูว่า DuckDB เลือกใช้ join algorithm แบบใดสำหรับ query นี้
print(con.execute("""
EXPLAIN SELECT l.loan_id, m.name FROM loan l JOIN member m ON l.member_id = m.member_id
""").fetchall()[0][1])


┌───────────────────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│   member_id = member_id   │              │
│                           │              │
│          ~20 Rows         │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         SEQ_SCAN          ││         SEQ_SCAN          │
│    ────────────────────   ││    ────────────────────   │
│        Table: loan        ││       Table: member       │
│   Type: Sequential Scan   ││   Type: Sequential Scan   │
│                           ││                           │
│        Projections:       ││        Projections:       │
│         member_id         ││         member_id         │
│          loan_id          ││            name           │
│                           ││                           │
│                           ││   Fi

ก่อนจะ JOIN ได้ ฐานข้อมูลต้องอ่านทั้งตาราง loan และทั้งตาราง member ทีละแถวจนครบ แล้วค่อยเอาผลมาจับคู่กันผ่าน HASH_JOIN

### HASH_JOIN ทำงานอย่างไร

HASH_JOIN ทำงานเป็น 2 ขั้นตอนหลัก — **Build phase** และ **Probe phase**

#### ขั้นตอนที่ 1: Build Phase (สร้าง Hash Table)

ฐานข้อมูลเลือก **ตารางที่เล็กกว่า** มาสร้าง hash table ก่อน (ในตัวอย่างของเรา `member` มี 12 แถว เล็กกว่า `loan` ที่มี 20 แถว)

สำหรับทุกแถวใน `member` จะคำนวณ **hash** ของค่า `member_id` แล้วเก็บลงใน hash table แบบนี้:

Hash Table (สร้างจาก member):

hash(1) → {member_id: 1, name: 'สมชาย'}

hash(2) → {member_id: 2, name: 'สมหญิง'}

hash(3) → {member_id: 3, name: 'สมศักดิ์'}

การค้นหาใน hash table ทำได้เร็วมาก (เกือบ O(1)) เพราะ hash function จะพาไปยังตำแหน่งที่ถูกต้องเกือบจะทันที ไม่ต้องไล่ดูทีละรายการ

#### ขั้นตอนที่ 2: Probe Phase (ไล่จับคู่)

จากนั้นฐานข้อมูลไล่อ่านตารางที่ใหญ่กว่า (`loan`) **ทีละแถว** แล้วเอาค่า `member_id` ของแต่ละแถวไปคำนวณ hash แล้ว "ถาม" hash table ทันทีว่ามีคู่ตรงกันไหม:

loan แถวที่ 1: member_id = 1  →  hash(1) → เจอ! คู่กับ 'สมชาย'  →  ได้ผลลัพธ์

loan แถวที่ 3: member_id = 2  →  hash(2) → เจอ! คู่กับ 'สมหญิง' →  ได้ผลลัพธ์

ไม่ต้องเทียบทุกแถวของ `loan` กับทุกแถวของ `member` (แบบ nested loop) แค่คำนวณ hash แล้วเช็คตำแหน่งเดียวก็รู้ผลทันที

#### ทำไมถึงเร็วกว่า Nested Loop

| วิธี | จำนวนการเปรียบเทียบ (โดยประมาณ) |
|---|---|
| Nested Loop | 12 × 20 = 240 ครั้ง (เทียบทุกคู่) |
| Hash Join | 12 (build) + 20 (probe) = 32 ครั้ง |

ยิ่งตารางใหญ่ขึ้น ความต่างยิ่งเห็นชัด — Hash Join จึงเป็นตัวเลือกที่ DuckDB มักเลือกใช้เมื่อ**ไม่มี index** และข้อมูล**ไม่ได้เรียงลำดับ** เหมือนในตัวอย่างของเรา

#### ข้อจำกัด

- ต้องมีพื้นที่หน่วยความจำพอสร้าง hash table ได้ (ถ้าตารางใหญ่ทั้งคู่ อาจมีปัญหาเรื่อง memory)
- ใช้ได้ดีกับ **equi-join** (เงื่อนไข `=`) เท่านั้น — Non-equi join (ใช้ `BETWEEN`) จะใช้ Hash Join แบบนี้ไม่ได้ เพราะ hash คำนวณได้เฉพาะค่าที่เท่ากัน ต้องใช้ algorithm อื่นแทน (เช่น nested loop)

---
# 5. เทคนิค JOIN ขั้นสูง

- **Self-join** — JOIN ตารางกับตัวเอง ด้วย alias คนละชื่อ ใช้เมื่อข้อมูลที่สัมพันธ์กันอยู่ในตารางเดียว
- **Non-equi join** — เงื่อนไข ON ไม่จำเป็นต้องเป็น `=` เสมอ ใช้ `>`, `<`, `BETWEEN` ได้
- **Multi-condition join** — ใส่หลายเงื่อนไขใน ON ด้วย `AND` เมื่อต้องจับคู่ด้วยหลายคอลัมน์


### 5.1 Self-join — หาคู่สมาชิกที่อยู่ใน city เดียวกัน

In [ ]:
# JOIN member กับตัวเอง — a.member_id < b.member_id ป้องกันจับคู่ตัวเองและคู่ซ้ำ
run("""
SELECT a.name AS member1, b.name AS member2, a.city AS city1, b.city AS city2
FROM member a
JOIN member b
  ON a.city = b.city
 AND a.member_id < b.member_id
""")


,member1,member2,city1,city2
0,สมชาย,สมศักดิ์,กรุงเทพฯ,กรุงเทพฯ
1,สมหญิง,สมศักดิ์,กรุงเทพฯ,กรุงเทพฯ
2,สมศรี,สมพร,เชียงใหม่,เชียงใหม่
3,วิชัย,อรุณ,ชลบุรี,ชลบุรี
4,พรทิพย์,ธนกร,ขอนแก่น,ขอนแก่น
5,สมชาย,สมหญิง,กรุงเทพฯ,กรุงเทพฯ


### 5.2 Non-equi join — จับคู่ตามช่วงเวลา

In [ ]:
run("""
SELECT *
FROM loan LIMIT 5
""")

,loan_id,member_id,book_id,loan_date,status
0,1,1,101,2024-01-05,returned
1,2,1,102,2024-02-10,returned
2,3,2,101,2024-01-20,returned
3,4,2,103,2024-03-01,borrowed
4,5,3,101,2024-04-15,returned


In [ ]:
# ตารางอ้างอิงช่วงเวลาภาคเรียน
con.execute("CREATE TABLE semester (semester_name VARCHAR, start_date DATE, end_date DATE)")
con.execute("""INSERT INTO semester VALUES
    ('ภาคเรียนที่ 1/2024', '2024-01-01', '2024-04-30'),
    ('ภาคเรียนที่ 2/2024', '2024-05-01', '2024-08-31')
""")

In [ ]:
# Non-equi join: จับคู่ตาม "ช่วง" ไม่ใช่ค่าที่เท่ากันเป๊ะ
run("""
SELECT s.semester_name, COUNT(*) AS loan_count
FROM loan l
JOIN semester s
  ON l.loan_date BETWEEN s.start_date AND s.end_date
GROUP BY s.semester_name
""")


,semester_name,loan_count
0,ภาคเรียนที่ 1/2024,13
1,ภาคเรียนที่ 2/2024,7


### 5.3 Multi-condition join — จับคู่ด้วยหลายคอลัมน์

In [ ]:
# ตาราง waitlist (รายการจอง) — เช็คว่าคำขอจองแต่ละรายการกลายเป็นการยืมจริงหรือไม่
con.execute("CREATE TABLE waitlist (request_id INTEGER, member_id INTEGER, book_id INTEGER, request_date DATE)")
con.execute("""INSERT INTO waitlist VALUES
    (1, 1, 101, '2023-12-20'),
    (2, 2, 105, '2024-01-15'),
    (3, 3, 104, '2024-04-10'),
    (4, 6, 107, '2024-03-01')
""")

In [ ]:
# ต้องจับคู่ทั้ง member_id และ book_id พร้อมกัน (2 เงื่อนไขใน ON)
run("""
SELECT w.request_id, m.name, b.title, l.loan_id
FROM waitlist w
JOIN member m ON w.member_id = m.member_id
JOIN book b ON w.book_id = b.book_id
JOIN loan l
  ON w.member_id = l.member_id
 AND w.book_id = l.book_id
""")


,request_id,name,title,loan_id
0,1,สมชาย,คิดแบบผู้ชนะ,1
1,3,สมศักดิ์,นิยายรักในสวนดอกไม้,6
2,4,วิชัย,จิตวิทยาการทำงาน,10


### แบบฝึกหัด 5
1. Self-join: หาคู่สมาชิกที่เคยยืมหนังสือเล่มเดียวกัน (ต้อง JOIN `member` กับตัวเอง ผ่าน `loan` สองครั้ง — แสดงชื่อทั้งคู่และชื่อหนังสือ)
2. Non-equi join: สร้างตาราง `quarter` แบ่งปี 2024 เป็น 4 ไตรมาส (Q1: ม.ค.-มี.ค., Q2: เม.ย.-มิ.ย., Q3: ก.ค.-ก.ย., Q4: ต.ค.-ธ.ค.) แล้วนับจำนวนการยืมในแต่ละไตรมาส
3. Multi-condition join: แก้ query ข้อ 5.3 ให้กรองเฉพาะคำขอจองที่การยืมจริงมีสถานะ `'returned'` เท่านั้น (เพิ่มเงื่อนไขที่ 3 ใน ON)


In [ ]:
# เขียนโค้ดข้อ 1 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 1

,member1,member2,title
0,สมชาย,สมพร,คิดแบบผู้ชนะ
1,สมหญิง,สมพร,คิดแบบผู้ชนะ
2,สมหญิง,ธนกร,เศรษฐศาสตร์แบบเข้าใจง่าย
3,สมศักดิ์,สมพร,คิดแบบผู้ชนะ
4,สมศรี,พรทิพย์,คู่มือ Data Science
5,สมศรี,สมพร,คิดแบบผู้ชนะ
6,สมพร,ธนกร,ประวัติศาสตร์ไทย
7,สมชาย,อรุณ,คิดแบบผู้ชนะ
8,สมศักดิ์,อรุณ,คิดแบบผู้ชนะ
9,สมหญิง,วิชัย,สอน Python เบื้องต้น


In [ ]:
con.execute("CREATE TABLE quarter (q_name VARCHAR, start_date DATE, end_date DATE)")
con.execute("""INSERT INTO quarter VALUES
    ('Q1', '2024-01-01', '2024-03-31'), ('Q2', '2024-04-01', '2024-06-30'),
    ('Q3', '2024-07-01', '2024-09-30'), ('Q4', '2024-10-01', '2024-12-31')
""")

In [ ]:
# เขียนโค้ดข้อ 2 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 2

,q_name,loan_count
0,Q1,10
1,Q2,7
2,Q3,3


In [ ]:
# เขียนโค้ดข้อ 3 ตรงนี้


In [ ]:
# ตัวอย่างผลลัพธ์ข้อ 3

,request_id,name,title,loan_id,status
0,1,สมชาย,คิดแบบผู้ชนะ,1,returned


---
# สรุป Chapter 3

| หัวข้อ | แก่นที่ได้ |
|---|---|
| หลักการของ JOIN | เงื่อนไข ON จำเป็นเสมอ — ลืมแล้วได้ cross join ที่ผิดและช้า |
| INNER JOIN & หลายตาราง | คืนเฉพาะแถวที่จับคู่ได้ — เครื่องมือหลักสร้าง feature table |
| OUTER JOIN | LEFT/RIGHT/FULL เก็บแถวไม่มีคู่ไว้ด้วย (เติม NULL) — เครื่องมือหลักของ EDA |
| Join Algorithms | Nested loop / Hash / Sort-merge — ระบบเลือกให้อัตโนมัติ ดูผ่าน EXPLAIN |
| JOIN ขั้นสูง | Self-join, Non-equi join (BETWEEN/>/<), Multi-condition (AND หลายเงื่อนไข) |

### ประเด็นเชิง Data Science ที่ต้องจำ
- **INNER JOIN** คือขั้นตอนแรกที่ทำให้ข้อมูลจากหลายตารางกลายมาเป็น 1 ตารางที่พร้อมป้อนเข้าโมเดล (feature table)
- **OUTER JOIN + IS NULL** คือแหล่งกำเนิด NULL ที่พบบ่อยที่สุดในงานจริง และเป็นเทคนิคหลักของ EDA เพื่อหาข้อมูลที่ขาดหาย
- **Join algorithms** ช่วยเข้าใจว่าทำไม query ที่ join ตารางใหญ่บางทีช้า บางทีเร็ว
- **Self-join / Non-equi / Multi-condition** เป็นเทคนิคที่ใช้บ่อยในงานวิเคราะห์เชิงเปรียบเทียบและ feature engineering ขั้นสูง

---
*จบ Chapter 3*
